In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'

In [3]:
import pandas as pd
import numpy as np

In [4]:
# while using virtual env
#df=pd.read_csv("news_article_classification_data.csv")

#using colab kernel
df=pd.read_csv("drive/MyDrive/Dataset/news_article_classification_data.csv")
df.head()
# df1=pd.read_csv("news_multilabel_merged_cleaned.csv")
# df.head()

,title,text,labels,url,section,revised_labels,label_cat_list
0,Megan Davies obituary,"My partner, Megan Davies, who has died aged 67...","['Politics', 'Technology']",https://www.theguardian.com/politics/2025/dec/...,politics,"['Politics', 'Technology']","[1, 1, 0, 0, 0, 0, 0, 0]"
1,Nigel Farage told to apologise by 26 of his sc...,Nigel Farage has been told to apologise for hi...,"['Politics', 'Technology']",https://www.theguardian.com/politics/2025/dec/...,politics,"['Politics', 'Technology']","[1, 1, 0, 0, 0, 0, 0, 0]"
2,Dulwich college head responds to claims of tee...,Dulwich college’s headteacher has responded to...,"['Politics', 'Technology']",https://www.theguardian.com/politics/2025/dec/...,politics,"['Politics', 'Technology']","[1, 1, 0, 0, 0, 0, 0, 0]"
3,Badenoch criticises Farage over refusal to apo...,Kemi Badenoch has questioned why Nigel Farage ...,"['Politics', 'Technology']",https://www.theguardian.com/politics/2025/dec/...,politics,"['Politics', 'Technology']","[1, 1, 0, 0, 0, 0, 0, 0]"
4,Nigel Farage is wrong – victims don’t forget b...,Regarding Nigel Farage’s difficulty believing ...,"['Politics', 'Technology']",https://www.theguardian.com/politics/2025/dec/...,politics,"['Politics', 'Technology']","[1, 1, 0, 0, 0, 0, 0, 0]"


In [5]:
df['labels'].unique()

array(["['Politics', 'Technology']", "['Politics', 'Business']",
       "['Economy', 'Politics']", "['Sports', 'Politics']",
       "['International_Affairs', 'Politics']",
       "['Politics', 'Education']",
       "['International_Affairs', 'Technology']",
       "['International_Affairs', 'Business']",
       "['International_Affairs', 'Economy']",
       "['International_Affairs', 'Sports']",
       "['International_Affairs', 'Health']",
       "['International_Affairs', 'Energy']", "['Economy', 'Business']",
       "['Economy', 'Business', 'Politics']",
       "['Technology', 'Economy', 'Business']",
       "['Health', 'Economy', 'Business']",
       "['Crime', 'Economy', 'Business']", "['Business', 'Technology']",
       "['Economy', 'Technology']", "['Education', 'Technology']",
       "['Health', 'Technology']", "['Science', 'Technology']",
       "['Energy', 'Technology']", "['Science', 'Business']",
       "['Science', 'Politics']", "['Science', 'Economy']",
       "['Science

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 47587 entries, 0 to 47586
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   title           47587 non-null  object
 1   text            47587 non-null  object
 2   labels          47587 non-null  object
 3   url             47587 non-null  object
 4   section         47587 non-null  object
 5   revised_labels  47587 non-null  object
 6   label_cat_list  47587 non-null  object
dtypes: object(7)
memory usage: 2.5+ MB


In [7]:
df.describe()  

,title,text,labels,url,section,revised_labels,label_cat_list
count,47587,47587,47587,47587,47587,47587,47587
unique,47568,47587,49,47587,7,38,36
top,UK inflation: which goods and services have ch...,"Wild plants, animals, fungi and algae support ...","['International_Affairs', 'Politics']",https://www.theguardian.com/environment/2022/j...,business,"['International_Affairs', 'Politics']","[1, 0, 0, 0, 1, 0, 0, 0]"
freq,9,1,7697,1,9985,7697,7697


In [8]:
print(df.duplicated().sum()) 
print(df['text'].duplicated().sum()) 
print(df['title'].duplicated().sum())

0
0
19


## 1. Data Preprocessing & Setup
We will parse the labels and create columns for our three scenarios (Title, Text, Title+Text).


In [9]:
import ast
import torch
import re
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertTokenizer, DistilBertModel
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, hamming_loss
import numpy as np

In [10]:
# Convert string representation of list to actual Python list
df['label_cat_list'] = df['label_cat_list'].apply(ast.literal_eval)

In [11]:
df['label_cat_list'].dtype

dtype('O')

In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 47587 entries, 0 to 47586
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   title           47587 non-null  object
 1   text            47587 non-null  object
 2   labels          47587 non-null  object
 3   url             47587 non-null  object
 4   section         47587 non-null  object
 5   revised_labels  47587 non-null  object
 6   label_cat_list  47587 non-null  object
dtypes: object(7)
memory usage: 2.5+ MB


In [13]:
df['label_cat_list'].iloc[0]

[1, 1, 0, 0, 0, 0, 0, 0]

In [ ]:
def clean_text(text):

    # Convert to lowercase
    text = text.lower()
    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)
    # Remove URLs
    text = re.sub(r'http\S+|www\.\S+', '', text)
    # Remove special characters (keeping basic punctuation for BERT context)
    text = re.sub(r'[^a-zA-Z0-9\s.,!?;:-]', '', text)
    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text


In [ ]:
df['title_cleaned'] = df['title'].apply(clean_text)


Cleaning 'title' column...


In [16]:
df['title_cleaned']

,title_cleaned
0,megan davies obituary
1,nigel farage told to apologise by 26 of his sc...
2,dulwich college head responds to claims of tee...
3,badenoch criticises farage over refusal to apo...
4,nigel farage is wrong victims dont forget bull...
...,...
47582,fears environment bills could be sidelined ami...
47583,country diary: we should open our eyes to the ...
47584,poacher who took 60k of salmon and trout from ...
47585,victims of brazils worst environmental disaste...


In [ ]:
df['text_cleaned'] = df['text'].apply(clean_text)


Cleaning 'text' column...


In [18]:
# Create the Title + Text combination
df['combined'] = df['title_cleaned'] + " - " + df['text_cleaned']

In [19]:
# Number of labels
num_labels = len(df['label_cat_list'].iloc[0])
print(f"Number of categories: {num_labels}")

Number of categories: 8


In [20]:
df['label_cat_list'].dtype

dtype('O')

## 2. Tokenization and Dataset 


In [21]:
class NewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
        
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, item):
        text = str(self.texts[item])
        label = self.labels[item]
        
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )
        
        return {
            'text': text,
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.float)
        }

tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

## 3. DistilBERT Multi-Label Model(sample)
We add a linear layer with Sigmoid activation on top of the DistilBERT encoder.


In [ ]:
import torch.nn as nn

class DistilBertMultiLabelClassifier(nn.Module):
    def __init__(self, n_classes):
        super(DistilBertMultiLabelClassifier, self).__init__()
        self.distilbert = DistilBertModel.from_pretrained('distilbert-base-uncased')
        self.dropout = nn.Dropout(0.3)
        self.classifier =nn.Sequential(
            nn.Linear(self.distilbert.config.hidden_size, 512),
            nn.ReLU(),
            
            nn.Dropout(0.2),
            nn.Linear(512,n_classes)
        )
         
    def forward(self, input_ids, attention_mask):
        output = self.distilbert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = output.last_hidden_state[:, 0]  
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)
        return logits


## 4. Data Preparation

In [ ]:
sample_df = df.sample(n=5000, random_state=42)
text_column = 'combined'

X_train, X_test, y_train, y_test = train_test_split(
    sample_df[text_column], 
    sample_df['label_cat_list'], 
    test_size=0.2, 
    random_state=42
)
print(f"Data Split Complete. Training set size: {len(X_train)}")

Data Split Complete. Training set size: 4000


## 5. Setup DataLoaders

In [24]:
batch_size = 32

train_dataset = NewsDataset(X_train.to_numpy(), y_train.to_numpy(), tokenizer)
test_dataset = NewsDataset(X_test.to_numpy(), y_test.to_numpy(), tokenizer)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
print("DataLoaders are ready.")

DataLoaders are ready.


## 6. Model Initialization

In [25]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [26]:
model = DistilBertMultiLabelClassifier(num_labels).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## 7. Training Function

In [27]:
from sklearn.metrics import accuracy_score

def train_model(model, train_loader, criterion, optimizer, device, epochs=3, val_loader=None):
    print("Starting Training...")
    history = {'train_loss': [], 'train_acc': [], 'val_macro_f1': [], 'val_acc': []}
    
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        all_preds = []
        all_labels = []
        for batch in train_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            optimizer.zero_grad()
            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels)
            
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

            probs = torch.sigmoid(logits)
            preds = (probs > 0.5).int()
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
        train_acc = accuracy_score(all_labels, all_preds)
        avg_train_loss = total_loss / len(train_loader)
        
        history['train_loss'].append(avg_train_loss)
        history['train_acc'].append(train_acc)
        
        print(f"Epoch {epoch+1}/{epochs} - Avg Train Loss: {avg_train_loss:.4f} - Train Acc: {train_acc:.4f}")

        if val_loader is not None:
            # Evaluate using existing eval_model function
            val_results = eval_model(model, val_loader, device)
            history['val_macro_f1'].append(val_results.get('macro_f1', 0))
            history['val_acc'].append(val_results.get('accuracy', 0))
            
    return model, history


## 8. Evaluation Function

In [28]:
def eval_model(model, test_loader, device):
    from sklearn.metrics import f1_score, hamming_loss, accuracy_score
    print("\nEvaluating Model...")
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            logits = model(input_ids, attention_mask)
            probs = torch.sigmoid(logits)
            preds = (probs > 0.5).int()
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
    micro_f1 = f1_score(all_labels, all_preds, average='micro')
    macro_f1 = f1_score(all_labels, all_preds, average='macro')
    h_loss = hamming_loss(all_labels, all_preds)
    acc = accuracy_score(all_labels, all_preds)
    
    print(f"Accuracy:       {acc:.4f}")
    print(f"Micro F1-score: {micro_f1:.4f}")
    print(f"Macro F1-score: {macro_f1:.4f}")
    print(f"Hamming Loss:   {h_loss:.4f}")
    
    return {'accuracy': acc, 'micro_f1': micro_f1, 'macro_f1': macro_f1, 'hamming_loss': h_loss}


## 9. Execute Training and Evaluation

In [29]:
# Train the model with validation and save history
model, baseline_history = train_model(model, train_loader, criterion, optimizer, device, epochs=5, val_loader=test_loader)

# Final Evaluation
results = eval_model(model, test_loader, device)


Starting Training...
Epoch 1/5 - Avg Train Loss: 0.4629 - Train Acc: 0.1410

Evaluating Model...
Accuracy:       0.4110
Micro F1-score: 0.7511
Macro F1-score: 0.7087
Hamming Loss:   0.1191
Epoch 2/5 - Avg Train Loss: 0.2840 - Train Acc: 0.5292

Evaluating Model...
Accuracy:       0.5440
Micro F1-score: 0.8094
Macro F1-score: 0.8231
Hamming Loss:   0.0989
Epoch 3/5 - Avg Train Loss: 0.2174 - Train Acc: 0.6292

Evaluating Model...
Accuracy:       0.5920
Micro F1-score: 0.8143
Macro F1-score: 0.8272
Hamming Loss:   0.0964
Epoch 4/5 - Avg Train Loss: 0.1761 - Train Acc: 0.6977

Evaluating Model...
Accuracy:       0.5930
Micro F1-score: 0.8199
Macro F1-score: 0.8337
Hamming Loss:   0.0939
Epoch 5/5 - Avg Train Loss: 0.1422 - Train Acc: 0.7432

Evaluating Model...
Accuracy:       0.5950
Micro F1-score: 0.8176
Macro F1-score: 0.8330
Hamming Loss:   0.0966

Evaluating Model...
Accuracy:       0.5950
Micro F1-score: 0.8176
Macro F1-score: 0.8330
Hamming Loss:   0.0966


## 10. Save the Trained Model


In [30]:
import os

# 1. Save the Tokenizer
tokenizer.save_pretrained('saved_distilbert_tokenizer')

# 2. Save the Custom PyTorch Model Weights
torch.save(model.state_dict(), 'distilbert_multilabel_weights.pth')

print("Model and tokenizer successfully saved to disk!")

Model and tokenizer successfully saved to disk!


### Save to drive 

In [31]:
# 1. Save the Tokenizer
tokenizer.save_pretrained('/content/MyDrive/Models/saved_distilbert_tokenizer')

# 2. Save the Custom PyTorch Model Weights
torch.save(model.state_dict(), '/content/MyDrive/Models/distilbert_multilabel_weights.pth')

print("Model and tokenizer successfully saved to drive!")

Model and tokenizer successfully saved to drive!


In [32]:
from google.colab import files

## download the file if needed

In [ ]:
files.download('saved_distilbert_tokenizer')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import shutil
from google.colab import files

# 1. Zip the tokenizer folder
# This creates 'saved_distilbert_tokenizer.zip'
shutil.make_archive('saved_distilbert_tokenizer', 'zip', 'saved_distilbert_tokenizer')

# 2. Download the zip file to your local machine
files.download('saved_distilbert_tokenizer.zip')

## 11. Load the Model & Make Predictions

In [ ]:

loaded_tokenizer = DistilBertTokenizer.from_pretrained('/content/MyDrive/Models/saved_distilbert_tokenizer')

#re-initialize the model 
loaded_model = DistilBertMultiLabelClassifier(num_labels)

# Load the trained weights into the architecture
loaded_model.load_state_dict(torch.load('/content/MyDrive/Models/distilbert_multilabel_weights.pth', map_location=device))
loaded_model.to(device)
loaded_model.eval() # Set to evaluation mode

Loading saved tokenizer and model...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


DistilBertMultiLabelClassifier(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
     

In [ ]:
category_names = [
        'Politics', 'Technology', 'Business', 'Economy', 
        'International_Affairs', 'Education', 'Health', 'Science/Other'
    ]

In [ ]:

def predict_news_category(raw_text, model, tokenizer, device, threshold=0.5):
    
    category_names = [
        'Politics', 'Technology', 'Business', 'Economy', 
        'International_Affairs', 'Education', 'Health', 'Science/Other'
    ]
    
    cleaned_text = clean_text(raw_text)
    
    encoding = tokenizer(
        cleaned_text,
        add_special_tokens=True,
        max_length=256,
        padding='max_length',
        truncation=True,
        return_attention_mask=True,
        return_tensors='pt'
    )
    
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)
    
    with torch.no_grad():
        logits = model(input_ids, attention_mask)
        probs = torch.sigmoid(logits)
        predictions_array = (probs > threshold).int().cpu().numpy()[0]
        raw_confidence = probs.cpu().numpy()[0]
    
    predicted_labels = []
    for i, is_predicted in enumerate(predictions_array):
        if is_predicted == 1:
            cat_name = category_names[i] if i < len(category_names) else f"Category_{i}"
            predicted_labels.append(cat_name)
                
    return predicted_labels, raw_confidence


Article: 'The US Government passed a new law heavily regulating artificial intelligence and stock market trading.'
Predicted Categories: ['Politics', 'Business', 'Economy']
Raw Confidence Output: [0.918 0.1   0.667 0.597 0.041 0.014 0.006 0.012]


In [ ]:

custom_article = "The US Government passed a new law heavily regulating artificial intelligence and stock market trading."
labels, scores = predict_news_category(custom_article, loaded_model, loaded_tokenizer, device)

print(f"\nArticle: '{custom_article}'")
print(f"Predicted Categories: {labels}")
print(f"Raw Confidence Output: {np.round(scores, 3)}")

## Hyperparameter Tuning with Optuna


In [ ]:
! pip install optuna

In [ ]:
import optuna
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from torch.utils.data import DataLoader
import torch
from sklearn.metrics import f1_score, accuracy_score, hamming_loss
from sklearn.model_selection import train_test_split
import pandas as pd

In [ ]:
len(df)*(15/100)

In [ ]:
sample_df = df.sample(frac=0.15, random_state=42)
text_column = 'combined'

In [ ]:
X_tune_train, X_tune_val, y_tune_train, y_tune_val = train_test_split(
    sample_df[text_column], 
    sample_df['label_cat_list'], 
    test_size=0.2, 
    random_state=42
)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
num_labels = len(df['label_cat_list'].iloc[0])
print(f"Number of categories: {num_labels}")

In [ ]:
def objective(trial):

    model_name = trial.suggest_categorical("model_name", ["distilbert-base-uncased", "roberta-base"])
    lr = trial.suggest_float("lr", 1e-5, 8e-5, log=True)
    batch_size = trial.suggest_categorical("batch_size", [16, 32])
    weight_decay = trial.suggest_float("weight_decay", 1e-4, 1e-1, log=True)
    warmup_ratio = trial.suggest_float("warmup_ratio", 0.0, 0.2)
    max_length = trial.suggest_categorical("max_length", [128, 256])
    
    epochs = trial.suggest_categorical("epoch",[2,3,4]) 
    
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, 
        num_labels=num_labels,
        problem_type="multi_label_classification"
    ).to(device)
    
    num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\nStarting Trial {trial.number} | Model: {model_name} | Params: {num_params:,} | LR: {lr:.2e} | Batch: {batch_size} | MaxLen: {max_length} | epoch: {epochs}")
    
    trial.set_user_attr("num_params", num_params)
    
    tune_train_dataset = NewsDataset(X_tune_train.to_numpy(), y_tune_train.to_numpy(), tokenizer, max_len=max_length)
    tune_val_dataset = NewsDataset(X_tune_val.to_numpy(), y_tune_val.to_numpy(), tokenizer, max_len=max_length)
    
    tune_train_loader = DataLoader(tune_train_dataset, batch_size=batch_size, shuffle=True)
    tune_val_loader = DataLoader(tune_val_dataset, batch_size=batch_size, shuffle=False)
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    
    total_steps = len(tune_train_loader) * epochs
    num_warmup_steps = int(total_steps * warmup_ratio)
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=num_warmup_steps, num_training_steps=total_steps)
    
    for epoch in range(epochs):
        model.train()
        for batch in tune_train_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            optimizer.zero_grad()
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            loss.backward()
            optimizer.step()
            scheduler.step()
            
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for batch in tune_val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            probs = torch.sigmoid(outputs.logits)
            preds = (probs > 0.5).int()
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
    macro_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    micro_f1 = f1_score(all_labels, all_preds, average='micro', zero_division=0)
    acc = accuracy_score(all_labels, all_preds)
    h_loss = hamming_loss(all_labels, all_preds)
    
    trial.set_user_attr("accuracy", acc)
    trial.set_user_attr("micro_f1", micro_f1)
    trial.set_user_attr("hamming_loss", h_loss)
    
    print(f"--- Trial {trial.number} Complete ---")
    print(f"Macro F1: {macro_f1:.4f} | Micro F1: {micro_f1:.4f} | Acc: {acc:.4f} | Hamming: {h_loss:.4f}\n")
    
    return macro_f1

In [ ]:
study = optuna.create_study(
    study_name="transformer_tuning",
    storage="sqlite:///optuna_study.db",
    load_if_exists=True,
    direction="maximize"
)

In [ ]:
study.optimize(objective, n_trials=15)

print("\n" + "="*50)
print("             OPTIMIZATION SUMMARY            ")
print("="*50)
print("Best Macro F1 Score:", study.best_trial.value)
print("Best Trial Metrics:")
print(f"  Accuracy:      {study.best_trial.user_attrs.get('accuracy', 0):.4f}")
print(f"  Micro F1:      {study.best_trial.user_attrs.get('micro_f1', 0):.4f}")
print(f"  Hamming Loss:  {study.best_trial.user_attrs.get('hamming_loss', 0):.4f}")
print("\nBest Trial Hyperparameters:")
for key, value in study.best_trial.params.items():
    print(f"  {key}: {value}")

print("\n" + "="*50)
print("             ALL TRIALS HISTORY            ")
print("="*50)


In [ ]:
df_trials = study.trials_dataframe()
# Save to a physical file for direct use
df_trials.to_csv('optuna_hyperparameters_history.csv', index=False)
print("\n[+] Saved complete hyperparameter history to 'optuna_hyperparameters_history.csv'")
cols_to_show = ['number', 'state', 'value', 'params_model_name', 'params_lr', 'user_attrs_accuracy', 'user_attrs_micro_f1']
available_cols = [c for c in cols_to_show if c in df_trials.columns]
if len(available_cols) > 0:
    print(df_trials[available_cols].to_string(index=False))


In [ ]:
# uncomment below to execute
study = optuna.create_study(
    study_name="transformer_tuning",
    storage="sqlite:///optuna_study.db",
    load_if_exists=True,
    direction="maximize"
)
print("Starting Optuna hyperparameter sweep...")
study.optimize(objective, n_trials=10) # 5 trials to start with

print("\n--- BEST PARAMETERS FOUND ---")
print("Best Macro F1 Score:", study.best_trial.value)
for key, value in study.best_trial.params.items():
    print(f"  {key}: {value}")

## Final Model Training

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score, accuracy_score, hamming_loss
import numpy as np

In [ ]:
# --- 3. FINAL MODEL INITIALIZATION & EXECUTION ---
import optuna
# Automatically pull the best parameters from the database!
loaded_study = optuna.load_study(study_name="transformer_tuning", storage="sqlite:///optuna_study.db")
best_params = loaded_study.best_trial.params

# Plug the optimized parameters dynamically straight into the final model
final_model_name = best_params.get("model_name", "roberta-base")
final_lr = best_params.get("lr", 5e-5)
final_batch_size = best_params.get("batch_size", 16)
final_weight_decay = best_params.get("weight_decay", 0.01)
final_warmup_ratio = best_params.get("warmup_ratio", 0.1)
final_max_length = best_params.get("max_length", 256)

final_epochs = 10  # Gave it 10 epochs since Early Stopping will halt it automatically
patience = 2

print(f"\n[System] Dynamically loaded best hyperparameters from DB:")
for k, v in best_params.items():
    print(f"  {k}: {v}")


In [ ]:
print(f"Initializing Tokenizer and Model: {final_model_name}")
final_tokenizer = AutoTokenizer.from_pretrained(final_model_name)
final_model = AutoModelForSequenceClassification.from_pretrained(
    final_model_name, 
    num_labels=num_labels,
    problem_type="multi_label_classification"
).to(device)

In [ ]:
#sample_df = df.sample(n=5000, random_state=42)
text_column = 'combined'

X_train, X_test, y_train, y_test = train_test_split(
    df[text_column], 
    df['label_cat_list'], 
    test_size=0.2, 
    random_state=42
)
print(f"Data Split Complete. Training set size: {len(X_train)}")

In [ ]:
print("Preparing Full-Size DataLoaders (100% of data)...")
final_train_dataset = NewsDataset(X_train.to_numpy(), y_train.to_numpy(), final_tokenizer, max_len=final_max_length)
final_val_dataset = NewsDataset(X_test.to_numpy(), y_test.to_numpy(), final_tokenizer, max_len=final_max_length)

final_train_loader = DataLoader(final_train_dataset, batch_size=final_batch_size, shuffle=True)
final_val_loader = DataLoader(final_val_dataset, batch_size=final_batch_size, shuffle=False)

In [ ]:
final_optimizer = torch.optim.AdamW(final_model.parameters(), lr=final_lr, weight_decay=final_weight_decay)

total_steps = len(final_train_loader) * final_epochs
num_warmup_steps = int(total_steps * final_warmup_ratio)
final_scheduler = get_linear_schedule_with_warmup(
    final_optimizer, num_warmup_steps=num_warmup_steps, num_training_steps=total_steps
)

In [ ]:
# --- 2. EARLY STOPPING TRAINING LOOP ---
def final_train_model(model, train_loader, val_loader, optimizer, scheduler, device, epochs=10, patience=2):
    best_macro_f1 = 0.0
    patience_counter = 0
    best_model_state = None
    
    print("\nStarting Training with Early Stopping...")
    for epoch in range(epochs):
        model.train()
        total_train_loss = 0
        
        for batch in train_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            optimizer.zero_grad()
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            loss.backward()
            
            optimizer.step()
            scheduler.step()
            total_train_loss += loss.item()
            
        avg_train_loss = total_train_loss / len(train_loader)
        
        # Evaluate
        val_metrics = final_eval_model(model, val_loader, device)
        
        print(f"\nEpoch {epoch+1}/{epochs}")
        print(f"  Train Loss : {avg_train_loss:.4f} | Val Loss : {val_metrics['loss']:.4f}")
        print(f"  Macro F1   : {val_metrics['macro_f1']:.4f} | Micro F1 : {val_metrics['micro_f1']:.4f}")
        print(f"  Accuracy   : {val_metrics['accuracy']:.4f} | Hamming  : {val_metrics['hamming_loss']:.4f}")
        
        # Early Stopping Logic (Maximize Macro F1)
        if val_metrics['macro_f1'] > best_macro_f1:
            best_macro_f1 = val_metrics['macro_f1']
            patience_counter = 0
            # Keep a backup of the absolute best weights
            best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()} 
            print("  --> Best model updated!")
        else:
            patience_counter += 1
            print(f"  --> No improvement. Patience: {patience_counter}/{patience}")
            
        if patience_counter >= patience:
            print("\n!!! EARLY STOPPING TRIGGERED !!!")
            break
            
    print(f"\nTraining Complete. Restoring best model with Macro F1: {best_macro_f1:.4f}")
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
    return model, history

In [ ]:
# --- 1. FINAL EVALUATION FUNCTION ---
def final_eval_model(model, dataloader, device):
    model.eval()
    all_preds = []
    all_labels = []
    total_loss = 0
    
    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            total_loss += outputs.loss.item()
            
            probs = torch.sigmoid(outputs.logits)
            preds = (probs > 0.5).int()
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
    val_loss = total_loss / len(dataloader)
    macro_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    micro_f1 = f1_score(all_labels, all_preds, average='micro', zero_division=0)
    acc = accuracy_score(all_labels, all_preds)
    h_loss = hamming_loss(all_labels, all_preds)
    
    return {
        'loss': val_loss,
        'macro_f1': macro_f1,
        'micro_f1': micro_f1,
        'accuracy': acc,
        'hamming_loss': h_loss
    }

In [ ]:
# 4. RUN PIPELINE
final_model = final_train_model(
    final_model, 
    final_train_loader, 
    final_val_loader, 
    final_optimizer, 
    final_scheduler, 
    device, 
    epochs=final_epochs, 
    patience=patience
)

## Save Optimized Model & Tokenizer
The trained state of your Optuna-optimized model is saved below for future deployment or inference use.

In [ ]:
import os

os.makedirs('saved_optimized_model', exist_ok=True)

# 1. Save Tokenizer
final_tokenizer.save_pretrained('saved_optimized_model/sample_tokenizer',local_files_only=True)

# 2. Save Model Weights
# Saving natively with HuggingFace allows skipping raw PyTorch arch re-initialization!
final_model.save_pretrained('saved_optimized_model/sample_model',local_files_only=True)

print("Optuna-optimized model and tokenizer successfully saved to 'saved_optimized_model/' directory!")


## Prediction Function
Load your automatically saved HuggingFace model and test it against a completely unseen string of text!

In [ ]:
import numpy as np

# --- 1. LOAD THE SAVED ASSETS ---
#rint("Loading saved tokenizer and optimized model...")
loaded_final_tokenizer = AutoTokenizer.from_pretrained('saved_optimized_model/sample_tokenizer')
loaded_final_model = AutoModelForSequenceClassification.from_pretrained('saved_optimized_model/sample_model')
loaded_final_model.to(device)
loaded_final_model.eval()

# --- 2. PREDICTION FUNCTION ---
def predict_custom_news(raw_text, model, tokenizer, device, threshold=0.5):
    # Ensure this strictly matches the columns exactly as they appeared in training!
    category_names = [
        'Politics', 'Technology', 'Business', 'Economy', 
        'International_Affairs', 'Education', 'Health', 'Science/Other'
    ]
    
    # Applies your existing cleaning logic
    try:
        cleaned_text = clean_text(raw_text)
    except NameError:
        print("clean_text function not found, passing raw text.")
        cleaned_text = raw_text
    
    encoding = tokenizer(
        cleaned_text,
        add_special_tokens=True,
        max_length=256,
        padding='max_length',
        truncation=True,
        return_attention_mask=True,
        return_tensors='pt'
    )
    
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)
    
    with torch.no_grad():
        # Access .logits because it is a HuggingFace Automodel
        logits = model(input_ids, attention_mask).logits
        probs = torch.sigmoid(logits)
        predictions_array = (probs > threshold).int().cpu().numpy()[0]
        raw_confidence = probs.cpu().numpy()[0]
    
    predicted_labels = []
    for i, is_predicted in enumerate(predictions_array):
        if is_predicted == 1:
            cat_name = category_names[i] if i < len(category_names) else f"Category_{i}"
            predicted_labels.append(cat_name)
                
    return predicted_labels, raw_confidence

# --- 3. TEST IT ---
custom_article = "Google DeepMind and OpenAI recently published new peer reviewed research regarding artificial intelligence scalability algorithms and quantum computing efficiency."
labels, scores = predict_custom_news(custom_article, final_model,final_tokenizer,device)

print(f"\nArticle: '{custom_article}'")
print(f"Predicted Categories: {labels}")
print(f"Raw Confidence Output: {np.round(scores, 3)}")


## Error Analysis (Predictions vs Reality)
This cell generates a detailed DataFrame comparing exactly what the model predicted alongside the true labels, and exports it to a CSV file. Use this to find patterns in where the model gets confused!

In [ ]:
import pandas as pd
import numpy as np
import torch

print("Running Error Analysis on Validation Data...")
loaded_final_model.eval()  # Make sure we use the optimized loaded model

all_probs = []
all_preds = []
all_true = []

# We use the final_test_loader because it has shuffle=False, meaning it perfectly lines up with X_test!
with torch.no_grad():
    for batch in final_test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels']
        
        logits = loaded_final_model(input_ids, attention_mask).logits
        probs = torch.sigmoid(logits)
        preds = (probs > 0.5).int()
        
        all_probs.extend(probs.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_true.extend(labels.numpy())

category_names = [
    'Politics', 'Technology', 'Business', 'Economy', 
    'International_Affairs', 'Education', 'Health', 'Science/Other'
]

def decode_labels(binary_array):
    return [category_names[i] for i, val in enumerate(binary_array) if val == 1]

# Map back to human readable text
true_text_labels = [decode_labels(arr) for arr in all_true]
pred_text_labels = [decode_labels(arr) for arr in all_preds]

# Create the Analysis DataFrame
analysis_df = pd.DataFrame({
    'Text': X_test.values,         # The raw input text
    'True_Labels': [', '.join(labels) for labels in true_text_labels],
    'Predicted_Labels': [', '.join(labels) for labels in pred_text_labels],
    'Raw_True_Array': list(all_true),
    'Raw_Pred_Array': list(all_preds),
    'Raw_Probabilities': [np.round(p, 3) for p in all_probs]
})

# Add a boolean column marking if it was perfectly accurate
analysis_df['Perfect_Match'] = analysis_df['True_Labels'] == analysis_df['Predicted_Labels']

analysis_df.to_csv('predictions_analysis.csv', index=False)
print("Analysis saved to 'predictions_analysis.csv'!")

# Display the exact articles the model got wrong
mistakes_df = analysis_df[analysis_df['Perfect_Match'] == False]

print(f"\nOverall Exact Match Accuracy: {analysis_df['Perfect_Match'].mean()*100:.2f}%")
print(f"{len(mistakes_df)} mistakes out of {len(analysis_df)} articles.")
print("\nHere are some of the misclassified examples:")
display(mistakes_df[['Text', 'True_Labels', 'Predicted_Labels']].head(10))


## Training Visualization & Learning Curves
This cell automatically visualizes the training health of your final model (Loss and F1 Score mapping) alongside interactive Optuna graphs to show which hyperparameters mattered the most.

In [ ]:
import matplotlib.pyplot as plt
import optuna

# 1. Plot the Learning Curves (Baseline DistilBERT vs Final Optimized Model)
try:
    plt.figure(figsize=(14, 6))
    
    # Plot 1: Train Loss Comparison
    plt.subplot(1, 2, 1)
    if 'baseline_history' in locals():
        plt.plot(baseline_history['train_loss'], label='Baseline Train Loss', linestyle='dotted', color='lightblue', marker='s')
    if 'final_history' in locals():
        plt.plot(final_history['train_loss'], label='Optimized Train Loss', marker='o', color='blue')
        plt.plot(final_history['val_loss'], label='Optimized Val Loss', marker='o', color='red')
    
    plt.title('Loss vs Training Epochs')
    plt.xlabel('Epochs')
    plt.ylabel('BCE Loss')
    plt.legend()
    plt.grid(True)
    
    # Plot 2: Macro F1 Score Over Time
    plt.subplot(1, 2, 2)
    if 'baseline_history' in locals():
        plt.plot(baseline_history['val_macro_f1'], label='Baseline Validation F1', color='lightgreen', marker='s', linestyle='dotted')
    if 'final_history' in locals():
        plt.plot(final_history['val_macro_f1'], label='Optimized Validation F1', color='green', marker='o', linestyle='dashed')
        
    plt.title('Validation F1 Score vs Epochs')
    plt.xlabel('Epochs')
    plt.ylabel('Macro F1 Score')
    plt.legend()
    plt.grid(True)
    
    plt.tight_layout()
    plt.show()

except Exception as e:
    print("[!] Could not plot final learning curves.", e)

# 2. Plot Optuna Hyperparameter Importance & History (Interactive)
try:
    # Pull the memory state directly from the local SQLite database!
    viz_study = optuna.load_study(study_name="transformer_tuning", storage="sqlite:///optuna_study.db")
    
    print("\n--- OPTUNA INTERACTIVE VISUALIZATIONS ---")
    print("Note: You can hover and interact with these plots!")
    
    # Plot 1: Show the progression of trials
    fig1 = optuna.visualization.plot_optimization_history(viz_study)
    fig1.show(renderer='colab') 
    
    # Plot 2: Show a mathematical breakdown of which hyperparameter impacted the accuracy the most
    fig2 = optuna.visualization.plot_param_importances(viz_study)
    fig2.show(renderer='colab')
    
except Exception as e:
    print("[!] Could not plot Optuna visualizations. Have you completed at least 1 trial in the Optuna cell?", e)
